# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from tqdm.notebook import tqdm
import contextlib
import joblib as joblib
import warnings

In [2]:
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [3]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = X.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.hour
        df['dayofweek'] = df['timestamp'].dt.weekday
        df = df.drop(columns='timestamp')

        return df

In [4]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target):
        self.target = target
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
        self.categorical = []

    def fit(self, X, y=None):
        self.categorical = X.select_dtypes(include=['object', 'category']).columns.tolist()
        if self.target in self.categorical:
            self.categorical.remove(self.target)
        self.encoder.fit(X[self.categorical])

        return self

    def transform(self, X):
        df = X.copy()
        y = df[self.target]
        df = df.drop(columns=self.target)

        encoded = self.encoder.transform(df[self.categorical])
        encoded = pd.DataFrame(
            encoded,
            columns=self.encoder.get_feature_names(self.categorical),
            index=df.index
        )

        df = df.drop(columns=self.categorical)
        df = pd.concat([df, encoded], axis=1)

        return df, y

In [5]:
class TrainValidationTest(BaseEstimator, TransformerMixin):
    def __init__(self, test_size=0.2, random_state=21):
        self.test_size = test_size
        self.random_state = random_state

    def fit(self, X, y=None):
        return self

    def transform(self, X, y):
        X_train_valid, X_test, y_train_valid, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y
        )

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_valid, y_train_valid,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y_train_valid
        )

        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [6]:
class ModelSelection():
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_estimators_ = {}
        self.results_ = []

    def choose(self, X_train, y_train, X_valid, y_valid):
        best_valid_score = -1
        best_model_name = None

        for idx, gs in enumerate(self.grids):
            model_name = self.grid_dict[idx]
            print(f'Estimator: {model_name}')

            param_grid = list(ParameterGrid(gs.param_grid))
            best_train_score = -1
            best_params = None

            for params in tqdm(param_grid):
                estimator = clone(gs.estimator).set_params(**params)
                scores = cross_val_score(
                    estimator, X_train, y_train,
                    cv=gs.cv, scoring=gs.scoring, n_jobs=gs.n_jobs
                )
                mean_score = scores.mean()

                if mean_score > best_train_score:
                    best_train_score = mean_score
                    best_params = params

            best_estimator = clone(gs.estimator).set_params(**best_params)
            best_estimator.fit(X_train, y_train)
            valid_score = best_estimator.score(X_valid, y_valid)

            print(f'Best params: {best_params}')
            print(f'Best training accuracy: {best_train_score:.3f}')
            print(f'Validation set accuracy score for best params: {valid_score:.3f} \n')

            self.results_.append({
                'model': model_name,
                'params': best_params,
                'valid_score': valid_score
            })

            self.best_estimators_[model_name] = best_estimator

            if valid_score > best_valid_score:
                best_valid_score = valid_score
                best_model_name = model_name

        print(f'Classifier with best validation set accuracy: {best_model_name}')
        return best_model_name

    def best_results(self):
        return pd.DataFrame(self.results_, columns=['model', 'params', 'valid_score'])

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [7]:
class Finalize():
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        print(f'Accuracy of the final model is {accuracy}')
        return accuracy

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print('Model was successfully saved')

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [8]:
df = pd.read_csv('../data/checker_submits.csv')
df.head()

,uid,labname,numTrials,timestamp
0,user_4,project1,1,2020-04-17 05:19:02.744528
1,user_4,project1,2,2020-04-17 05:22:45.549397
2,user_4,project1,3,2020-04-17 05:34:24.422370
3,user_4,project1,4,2020-04-17 05:43:27.773992
4,user_4,project1,5,2020-04-17 05:46:32.275104


In [9]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('dayofweek'))
])

X, y = preprocessing.fit_transform(df)
X.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,uid_user_16,uid_user_17,uid_user_18,uid_user_19,uid_user_2,uid_user_20,uid_user_21,uid_user_22,uid_user_23,uid_user_24,uid_user_25,uid_user_26,uid_user_27,uid_user_28,uid_user_29,uid_user_3,uid_user_30,uid_user_31,uid_user_4,uid_user_6,uid_user_7,uid_user_8,labname_code_rvw,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [10]:
y.head()

0    4
1    4
2    4
3    4
4    4
Name: dayofweek, dtype: int64

In [11]:
tvt = TrainValidationTest(test_size=0.2, random_state=21)
X_train, X_valid, X_test, y_train, y_valid, y_test = tvt.transform(X, y)

print(X_train.shape, X_valid.shape, X_test.shape)
print(y_train.shape, y_valid.shape, y_test.shape)

(1078, 43) (270, 43) (338, 43)
(1078,) (270,) (338,)


In [12]:
svm = SVC()
tree = DecisionTreeClassifier()
rf = RandomForestClassifier()

svm_params = [{
    'kernel': ('linear', 'rbf', 'sigmoid'),
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ('balanced', None),
    'random_state': [21],
    'probability': [True]
}]

tree_params = [{
    'max_depth': range(1, 25),
    'class_weight': ('balanced', None),
    'criterion': ('entropy', 'gini'),
    'random_state': [21]
}]

rf_params = [{
    'n_estimators': [5, 10, 50, 100],
    'max_depth': range(1, 31),
    'class_weight': ('balanced', None),
    'criterion': ('entropy', 'gini'),
    'random_state': [21]
}]

jobs = -1

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', n_jobs=jobs)
gs_tree = GridSearchCV(estimator=tree, param_grid=tree_params, scoring='accuracy', n_jobs=jobs)
gs_rf = GridSearchCV(estimator=rf, param_grid=rf_params, scoring='accuracy', n_jobs=jobs)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

In [13]:
model_selection = ModelSelection(grids, grid_dict)
best_model_name = model_selection.choose(X_train, y_train, X_valid, y_valid)

Estimator: SVM


  0%|          | 0/72 [00:00<?, ?it/s]

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.842
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree


  0%|          | 0/96 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 23, 'random_state': 21}
Best training accuracy: 0.852
Validation set accuracy score for best params: 0.859 

Estimator: Random Forest


  0%|          | 0/480 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 28, 'n_estimators': 100, 'random_state': 21}
Best training accuracy: 0.897
Validation set accuracy score for best params: 0.889 

Classifier with best validation set accuracy: Random Forest


In [14]:
results_df = model_selection.best_results()
results_df

,model,params,valid_score
0,SVM,"{'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}",0.877778
1,Decision Tree,"{'class_weight': None, 'criterion': 'gini', 'max_depth': 23, 'random_state': 21}",0.859259
2,Random Forest,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 28, 'n_estimators': 100, 'random_state': 21}",0.888889


In [15]:
best_estimator = model_selection.best_estimators_[best_model_name]

final = Finalize(best_estimator)
accuracy = final.final_score(X_train, y_train, X_test, y_test)

model_filename = f"{best_model_name.replace(' ', '_')}_{accuracy}.sav"
final.save_model(f'../data/{model_filename}')

Accuracy of the final model is 0.9142011834319527
Model was successfully saved


In [16]:
loaded = joblib.load(f'../data/{model_filename}')
final_loaded = Finalize(loaded)
accuracy_loaded = final_loaded.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model is 0.9142011834319527


In [17]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('labname'))
])

X, y = preprocessing.fit_transform(df)
y.head()

0    project1
1    project1
2    project1
3    project1
4    project1
Name: labname, dtype: object

In [18]:
X.head()

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,uid_user_16,uid_user_17,uid_user_18,uid_user_19,uid_user_2,uid_user_20,uid_user_21,uid_user_22,uid_user_23,uid_user_24,uid_user_25,uid_user_26,uid_user_27,uid_user_28,uid_user_29,uid_user_3,uid_user_30,uid_user_31,uid_user_4,uid_user_6,uid_user_7,uid_user_8
0,1,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,3,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,4,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,5,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
